# Netflix: EDA каталога, оценок и модельных гипотез об аудитории

**Учебный групповой проект НИУ ВШЭ, 2025.** Команда: Анастас Валерия, Варнавская Анна, Дудин Даниил, Хусаинов Даниэль.

Воспроизводимая основная часть исследования дополнена каталогом из 7 787 записей, выборкой пользовательских оценок и агрегированными результатами ранее выполненной LLM-классификации. Последние — не данные о реальных зрителях.



## 1. Источник данных

В репозитории опубликован `netflix_shows.csv`, экспортированный из приложенного преподавателем `NetflixShows.xlsx`: **1 000 исходных строк и 7 колонок**. В рамках исходной командной работы использовались также `netflix_titles.csv`, `df_combined.csv` и `all_netflix.csv`; эти файлы не приложены, поэтому здесь не воспроизводятся.

`rating` — возрастная классификация шоу (например, TV-MA); `user rating score` — числовая пользовательская оценка. Эти переменные нельзя смешивать.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind
from IPython.display import display

sns.set_theme(style='whitegrid')
path = Path('netflix_shows.csv')
if not path.exists():
    raise FileNotFoundError('Поместите netflix_shows.csv рядом с Notebook')
raw = pd.read_csv(path)
required = {'title','rating','ratingLevel','ratingDescription','release year','user rating score','user rating size'}
assert required.issubset(raw.columns)
print('Исходных записей:', len(raw))
display(raw.head())

## 2. Качество данных и удаление дублей

Сравниваем точные повторы всех полей, долю пропусков и диапазоны числовых значений. Удаляем только полные дубли. Причина их возникновения **не доказана**: гипотеза о рекомендательном алгоритме, обсуждаемая в презентации, требует отдельной проверки.

In [ ]:
audit = pd.DataFrame({'type': raw.dtypes.astype(str), 'missing': raw.isna().sum(), 'missing_%': (raw.isna().mean()*100).round(1)})
display(audit)
print('Точных дублей:', raw.duplicated().sum())
print('Уникальных названий:', raw.title.nunique())
clean = raw.drop_duplicates().copy()
clean['release year'] = pd.to_numeric(clean['release year'],errors='coerce')
clean['user rating score'] = pd.to_numeric(clean['user rating score'],errors='coerce')
print('После дедупликации:', len(clean))
print('Нет числовой оценки:', clean['user rating score'].isna().sum())

## 3. Возрастные категории контента

Показываем частоту категорий `rating` в очищенной **выборке**, а не в полном каталоге Netflix.

In [ ]:
counts = clean.rating.fillna('Не указано').value_counts().head(12)
fig,ax = plt.subplots(figsize=(10,5))
sns.barplot(x=counts.values,y=counts.index,color='#c6284f',ax=ax)
ax.set(title='12 наиболее частых возрастных категорий',xlabel='Число записей',ylabel='Категория')
plt.tight_layout()
plt.show()
display((clean.rating.value_counts(normalize=True)*100).round(1).head(10).rename('Доля, %').to_frame())

## 4. Пользовательские оценки

Описываем только шоу, для которых `user rating score` известен. **Пропуск не равен нулю.** Распределение оценённой части выборки не доказывает удовлетворённости всех зрителей Netflix.

In [ ]:
scored = clean.dropna(subset=['user rating score']).copy()
print('С известной оценкой:',len(scored),'из',len(clean))
display(scored['user rating score'].describe().round(2).to_frame('Оценка'))
fig,ax=plt.subplots(figsize=(10,4))
sns.histplot(scored['user rating score'],bins=20,color='#244977',edgecolor='white',ax=ax)
ax.axvline(scored['user rating score'].median(),linestyle='--',color='#c6284f',label='Медиана')
ax.set(title='Распределение числовых оценок',xlabel='User rating score',ylabel='Шоу')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Годы выпуска

Год создания контента не равен году его добавления в Netflix и **не позволяет** говорить о производстве Netflix Originals.

In [ ]:
year_counts = clean['release year'].dropna().astype(int).value_counts().sort_index()
fig,ax=plt.subplots(figsize=(11,4))
ax.bar(year_counts.index,year_counts.values,color='#244977')
ax.set(title='Шоу по году выхода',xlabel='Год выхода',ylabel='Уникальных записей')
plt.tight_layout()
plt.show()
display(year_counts.sort_values(ascending=False).head(8).rename('Число').to_frame())

## 6. Сравнение оценок выпусков 2016 и 2017 годов

Отвечаем на один из вопросов исходного задания. Разность средних описывает **только выбранные записи**, а статистический тест не позволяет называть один год более успешным для бизнеса Netflix. Используем Welch t-test как исследовательскую проверку, без причинных утверждений.

In [ ]:
selected = scored[scored['release year'].isin([2016,2017])].copy()
display(selected.groupby('release year')['user rating score'].agg(['count','mean','median','std']).round(2))
s16 = selected.loc[selected['release year']==2016,'user rating score']
s17 = selected.loc[selected['release year']==2017,'user rating score']
test = ttest_ind(s17,s16,equal_var=False)
print(f'Разность средних (2017−2016): {s17.mean()-s16.mean():.2f}')
print(f'Welch p-value: {test.pvalue:.4f}')
fig,ax=plt.subplots(figsize=(7,4))
sns.boxplot(data=selected,x='release year',y='user rating score',order=[2016,2017],color='#99adc6',ax=ax)
ax.set(title='Оценки по годам',xlabel='Год',ylabel='Числовая оценка')
plt.tight_layout()
plt.show()

## 7. Описательный портрет шоу: Friends

В исходной командной презентации приведён пример сериала «Friends». Ниже используем **только поля, подтверждённые исходным CSV**; сведения о сезонах и зрителях из других источников не добавляем.

In [ ]:
example = clean.loc[clean.title.astype(str).str.fullmatch('Friends',case=False,na=False)]
if example.empty:
    print('Friends не найден в предоставленном файле')
else:
    display(example[['title','rating','release year','user rating score']].drop_duplicates())
    valid = example['user rating score'].dropna()
    if len(valid):
        x=valid.iloc[0]
        print(f'Числовая оценка: {x:g}; процент оценённых шоу выборки с оценкой не выше: {(scored["user rating score"]<=x).mean()*100:.1f}%')

## 8. Итоги и ограничения

- В исходном файле **1 000 записей**, **500 точных дублей**, после очистки — **500 записей**. Различий между точными дублями нет, но причина их появления неизвестна.
- Числовая оценка указана лишь у части очищенной выборки; **пропуски нельзя интерпретировать как низкие оценки**.
- `rating` — возрастная категория, `user rating score` — оценка пользователей.
- Годы выхода не являются доказательством изменения стратегии Netflix или количества произведённых компанией фильмов.
- Сравнение 2016/2017 — описательный анализ, который требует внешней проверки репрезентативности выборки.

**Расширение оригинальной командной работы:** анализ жанров, стран и форматов по `netflix_titles.csv`, а также LLM-предположения о зрителях по `df_combined.csv`. Эти данные не были приложены, и в портфолио они не выдаются за воспроизведённые факты. LLM-классификация **не является измерением реальной аудитории**.

Работа основана на командном проекте №50 НИУ ВШЭ; индивидуальные доли участия не документированы.

## 9. Расширение: второй каталог Netflix

В дополнение к исходному `NetflixShows` используется более широкий каталог из 7 787 записей. Файлы `catalog_part_01.csv` … `catalog_part_08.csv` содержат только признаки, необходимые для EDA; их объединяют без изменения порядка строк. `release_year` означает **год выхода произведения**, а не год производства Netflix Originals или добавления на платформу.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

parts = sorted(Path('.').glob('catalog_part_*.csv'))
assert len(parts) == 8, 'Ожидалось восемь частей расширенного каталога'
catalog = pd.concat([pd.read_csv(p) for p in parts], ignore_index=True)
assert len(catalog) == 7787 and catalog.show_id.is_unique
print('Записей в каталоге:', len(catalog))
display(catalog['type'].value_counts().rename('Количество').to_frame())

## 10. Фильмы и сериалы по годам выхода

График показывает **число позиций в данном историческом наборе данных** по годам оригинального выхода; нельзя делать вывод о числе просмотров или производственной стратегии Netflix без дополнительных данных.

In [ ]:
year_type = catalog[catalog.release_year.between(2000,2020)].groupby(['release_year','type']).size().unstack(fill_value=0)
ax=year_type.plot(figsize=(11,4), linewidth=2)
ax.set(title='Распределение фильмов и сериалов по годам выхода', xlabel='Год выхода', ylabel='Число записей')
plt.tight_layout();plt.show()

## 11. Страны и жанры

В списках через запятую могут встречаться **несколько стран и жанров** для одного произведения. Поэтому количество упоминаний не эквивалентно числу уникальных проектов. Страны совместного производства учитываются отдельно.

In [ ]:
country = catalog.dropna(subset=['country']).assign(country=lambda d: d.country.str.split(',')).explode('country')
country['country'] = country['country'].str.strip()
genres = catalog.dropna(subset=['listed_in']).assign(genre=lambda d: d.listed_in.str.split(',')).explode('genre')
genres['genre'] = genres['genre'].str.strip()
fig,axes=plt.subplots(1,2,figsize=(13,5))
sns.barplot(x=country.country.value_counts().head(10).values,y=country.country.value_counts().head(10).index,ax=axes[0])
sns.barplot(x=genres.genre.value_counts().head(10).values,y=genres.genre.value_counts().head(10).index,ax=axes[1])
axes[0].set_title('Страны, топ-10 упоминаний'); axes[1].set_title('Жанры, топ-10 упоминаний')
plt.tight_layout();plt.show()

## 12. Дополнительная выборка с пользовательскими оценками

`rating_sample.csv` включает **320 строк, из них 126 уникальных по всем столбцам**. Из-за повторов и неизвестного отбора она не является репрезентативной выборкой пользователей Netflix.

In [ ]:
rating_sample = pd.read_csv('rating_sample.csv')
print('Строк до очистки:',len(rating_sample),'после удаления полных дублей:',len(rating_sample.drop_duplicates()))
display(rating_sample.drop_duplicates().groupby('type')['user rating score'].agg(['count','mean','median']).round(2))

## 13. LLM-классификация предполагаемой аудитории

В командной работе модель YandexGPT создавала метки предполагаемого возраста, условной гендерной ориентации и контекста просмотра по описаниям. Это **не фактическая аналитика пользователей**. Для прозрачности публикуются только **агрегированные результаты** исходного `df_combined.csv`, а не тексты промптов и необработанные ответы. Из 8 807 записей 438 имеют флаг ошибки, 1 020 идентификаторов не встречаются в прилагаемом расширенном каталоге. `confidence` модели не является калиброванной вероятностью.

In [ ]:
import json
with open('llm_summary.json',encoding='utf-8') as fh:
    summary = json.load(fh)
print('Размеры и ограничения:',summary['metadata'])
age = pd.Series(summary['age_counts']).sort_values(ascending=False)
fig,axs=plt.subplots(1,2,figsize=(13,4))
sns.barplot(x=age.values,y=age.index,ax=axs[0]); axs[0].set(title='Возрастные метки, присвоенные LLM',xlabel='Число классификаций',ylabel='Категория')
share = pd.DataFrame(summary['genre_age_counts']).set_index('genre')
share = share.div(share.sum(axis=1),axis=0)*100
sns.heatmap(share,annot=True,fmt='.0f',cmap='Blues',ax=axs[1]); axs[1].set(title='Модельные возрастные метки по жанрам, %',xlabel='Метка LLM',ylabel='Жанр')
plt.tight_layout();plt.show()

## 14. Выводы по расширенной части

- В историческом каталоге есть **5 377 фильмов** и **2 410 сериалов**; это структура выборки, не распределение зрительского спроса.
- География и жанры изучены как характеристики произведений, а не как доказательство коммерческой успешности.
- Выборка с оценками содержит большое количество повторов; для описательных сравнений используем уникальные строки.
- В результатах LLM чаще всего встречается метка `18–24` (4 855 записей), **но это не означает, что большинство реальных зрителей Netflix — 18–24-летние**.
- Для реальных продуктовых решений потребуются данные просмотров, удержания, подписок и проверка качества LLM-классификации.

**Работа командная.** Атрибуция сохранена. Новая структура предназначена для воспроизводимой презентации методов и аккуратного разделения наблюдаемых данных и модельных предположений.